# Parser Milestone 2 (Colab CPU-only: Full / Pilot / Resume)

Notebook chạy parser M2 trên corpus M1 đã khôi phục từ backup Google Drive, không gọi crawler arXiv hay Semantic Scholar.
Chọn **Runtime -> Change runtime type -> CPU** trước khi chạy. Nếu có sẵn file `/content/m2_run_config.json` trong workspace, notebook tự đọc tham số từ file đó; nếu chạy trực tiếp trên giao diện Colab thì chỉnh các giá trị trong ô cấu hình bên dưới.


In [ ]:
import json, os
from pathlib import Path

CONFIG_PATH = Path(os.environ.get('M2_RUN_CONFIG', '/content/m2_run_config.json'))
if not CONFIG_PATH.is_file():
    DEFAULT_CONFIG = {
        'member_id': '24120282',
        'short_name': 'DAT',
        'generation': '8bc30379e093456896d3f1d659e9b8ff',
        'head_sha256': '',
        'mode': 'full',  # 'pilot' | 'full' | 'resume'
        'pilot_size': 250,
        'pilot_seed': 20261121,
        'bundle_path': '/content/arxiv_m2_runtime.zip',
        'bundle_sha256': 'd69046aa4821aa94ba5a2403cbe847213252e7004563f1fc9db26e7dbf7843c1',
        'drive_backup_dir': '/content/drive/MyDrive/arxiv-harvester-data/team_full_20261115/backups/24120282',
        'chain_manifest_path': '/content/m2_chain.json',
        'drive_output_root': '/content/drive/MyDrive/arxiv-harvester-m2',
        'root_overrides_rel': 'configs/parser_roots.json',
        'sample_interval_sec': 0.5,
        'min_free_gb': 2.0,
        'sync_batch_interval_sec': 120.0,
        'fresh_output': True,
    }
    CONFIG_PATH.write_text(json.dumps(DEFAULT_CONFIG, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

cfg = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
print('Loaded config:', json.dumps({k: v for k, v in cfg.items() if 'token' not in k}, ensure_ascii=False, indent=2))


In [ ]:
from pathlib import Path
import os

if Path('/dev/nvidia0').exists() or os.environ.get('COLAB_TPU_ADDR'):
    raise RuntimeError('M2 yêu cầu runtime CPU-only; hãy đổi Runtime -> CPU rồi chạy lại.')

if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
print('Drive ready:', Path('/content/drive/MyDrive').exists())


In [ ]:
import os, subprocess, sys
from pathlib import Path

runner_path = Path('/content/m2_colab_runner.py')
if not runner_path.is_file():
    runner_path.write_text('"""Colab CPU-only orchestrator for Milestone 2 restore, audit, pilot, full, and resume."""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport resource\nimport shutil\nimport stat\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport urllib.request\nimport zipfile\n\n\ndef allowed_path(name: str) -> bool:\n    path = Path(name)\n    return (\n        path.as_posix() == name\n        and not path.is_absolute()\n        and ".." not in path.parts\n        and "\\\\" not in name\n        and ":" not in name\n        and (\n            name in {"pyproject.toml", "requirements.txt", "README.md", "LICENSE"}\n            or (name.startswith("src/") and name.endswith(".py"))\n            or (name.startswith("configs/") and path.suffix in {".json", ".md"})\n        )\n    )\n\n\ndef install_bundle(archive_path: str | Path, target_root: str | Path, expected_sha256: str) -> tuple[Path, dict]:\n    archive_path, target_root = Path(archive_path), Path(target_root).resolve()\n    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()\n    if not expected_sha256 or digest != expected_sha256:\n        raise ValueError(f"Bundle SHA-256 mismatch: expected {expected_sha256}, got {digest}")\n    with zipfile.ZipFile(archive_path) as archive:\n        entries = archive.infolist()\n        names = [row.filename for row in entries]\n        if len(names) != len(set(names)) or len(names) > 150 or sum(row.file_size for row in entries) > 8 * 1024**2:\n            raise ValueError("Invalid bundle structure or size")\n        manifest = json.loads(archive.read("bundle_manifest.json"))\n        if manifest.get("format_version") != 1 or manifest.get("runtime_protocol") != 1:\n            raise ValueError("Unsupported bundle runtime protocol")\n        if set(names) != set(manifest["files"]) | {"bundle_manifest.json"}:\n            raise ValueError("Bundle file list does not match manifest")\n        for row in entries:\n            if row.filename != "bundle_manifest.json" and not allowed_path(row.filename):\n                raise ValueError("File outside source allowlist: " + row.filename)\n            if stat.S_ISLNK(row.external_attr >> 16) or row.flag_bits & 1:\n                raise ValueError("Bundle contains symlink or encrypted entry")\n        contents = {name: archive.read(name) for name in names}\n        for name, expected in manifest["files"].items():\n            if hashlib.sha256(contents[name]).hexdigest() != expected:\n                raise ValueError("Bundle member SHA-256 mismatch: " + name)\n    target_root.mkdir(parents=True, exist_ok=True)\n    destination = target_root / digest[:16]\n    if destination.exists():\n        if destination.is_symlink() or not destination.resolve().is_relative_to(target_root):\n            raise ValueError("Runtime source path escapes target root")\n        expected_python = {name for name in manifest["files"] if name.startswith("src/") and name.endswith(".py")}\n        actual_python = {p.relative_to(destination).as_posix() for p in (destination / "src").rglob("*.py")}\n        if actual_python != expected_python:\n            raise ValueError("Existing runtime source contains unexpected Python files")\n        if any(\n            not (destination / name).is_file()\n            or (destination / name).is_symlink()\n            or hashlib.sha256((destination / name).read_bytes()).hexdigest() != expected\n            for name, expected in manifest["files"].items()\n        ):\n            raise ValueError("Installed source differs from bundle; use a clean source folder")\n    else:\n        with tempfile.TemporaryDirectory(prefix=".source-", dir=target_root) as temporary:\n            staging = Path(temporary).resolve()\n            unpacked = staging / "runtime"\n            for name, data in contents.items():\n                target = unpacked / name\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(data)\n            unpacked.rename(destination)\n    return destination, manifest\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef atomic_copy(src: Path, dst: Path) -> str:\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    digest = sha256_file(src)\n    tmp = dst.with_name(dst.name + ".tmp")\n    try:\n        shutil.copyfile(src, tmp)\n        if sha256_file(tmp) != digest:\n            raise OSError(f"Copy checksum mismatch for {dst}")\n        os.replace(tmp, dst)\n    finally:\n        tmp.unlink(missing_ok=True)\n    return digest\n\n\ndef atomic_write_json(path: Path, payload: object) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    try:\n        tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n        os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef download_drive_file(file_id: str, dest: Path, expected_sha256: str | None = None, retries: int = 5) -> None:\n    if dest.is_file() and expected_sha256 is not None and sha256_file(dest) == expected_sha256:\n        return\n    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"\n    headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) arxiv-harvester-m2-runner/1.0"}\n    last_err: Exception | None = None\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    tmp = dest.with_name(dest.name + ".tmp")\n    for attempt in range(retries):\n        try:\n            req = urllib.request.Request(url, headers=headers)\n            with urllib.request.urlopen(req, timeout=120) as resp, tmp.open("wb") as out:\n                while True:\n                    chunk = resp.read(1024 * 1024)\n                    if not chunk:\n                        break\n                    out.write(chunk)\n            if expected_sha256 is not None and sha256_file(tmp) != expected_sha256:\n                raise ValueError(f"Checksum mismatch for downloaded {dest.name} (file_id={file_id})")\n            os.replace(tmp, dest)\n            return\n        except Exception as exc:  # noqa: BLE001\n            last_err = exc\n            tmp.unlink(missing_ok=True)\n            time.sleep(min(15.0, 1.0 * (2**attempt)))\n    raise RuntimeError(f"Failed downloading {dest.name} ({file_id}): {last_err}")\n\n\ndef ensure_backup_dir(cfg: dict) -> tuple[Path, str]:\n    drive_dir = cfg.get("drive_backup_dir")\n    if drive_dir and (Path(drive_dir) / "HEAD.json").is_file():\n        return Path(drive_dir), "drive_mount"\n    chain_path = Path(cfg["chain_manifest_path"])\n    if not chain_path.is_file():\n        raise FileNotFoundError(f"Neither drive_backup_dir ({drive_dir}) nor chain_manifest_path ({chain_path}) is available")\n    chain = json.loads(chain_path.read_text(encoding="utf-8"))\n    mssv = cfg["member_id"]\n    generation = cfg["generation"]\n    local_backup = Path(f"/content/m2_backups/{mssv}/{generation}")\n    local_backup.mkdir(parents=True, exist_ok=True)\n    head_path = local_backup / "HEAD.json"\n    if not head_path.is_file() or json.loads(head_path.read_text(encoding="utf-8")).get("generation") != generation:\n        download_drive_file(chain["head_file_id"], head_path)\n    items = chain["oldest_to_newest"]\n    print(f"[runner] Downloading {len(items)} verified backup generations for {mssv}...", flush=True)\n    t0 = time.perf_counter()\n\n    def _fetch_one(item: dict) -> str:\n        gen = item["generation"]\n        download_drive_file(item["receipt_file_id"], local_backup / f"{gen}.json", item["receipt_sha256"])\n        download_drive_file(item["archive_file_id"], local_backup / f"{gen}.tar.gz", item["archive_sha256"])\n        return gen\n\n    done = 0\n    with ThreadPoolExecutor(max_workers=6) as pool:\n        futures = [pool.submit(_fetch_one, item) for item in items]\n        for fut in as_completed(futures):\n            fut.result()\n            done += 1\n            if done % 50 == 0 or done == len(items):\n                print(f"[runner] Backup download progress: {done}/{len(items)} ({time.perf_counter() - t0:.1f}s)", flush=True)\n    return local_backup, "public_chain_download"\n\n\ndef select_stratified_pilot(pub_path: Path, audit_path: Path, target_size: int, seed: int, out_dir: Path) -> tuple[Path, Path, Path]:\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    from src.crawler.storage import PUBLICATIONS_SCHEMA, export_to_course_spec, save_publications_parquet, save_references_parquet\n\n    pubs = {row["id"]: row for row in pq.ParquetFile(pub_path).read().to_pylist()}\n    audit_rows = pq.ParquetFile(audit_path).read().to_pylist()\n    by_paper: dict[str, list[dict]] = {}\n    for row in audit_rows:\n        by_paper.setdefault(row["arxiv_id"], []).append(row)\n\n    features = {}\n    for pid, p in pubs.items():\n        arows = by_paper.get(pid, [])\n        features[pid] = {\n            "num_versions": len(p.get("versions") or []),\n            "max_tex_files": max((r.get("tex_files") or 0 for r in arows), default=0),\n            "total_bbl_files": sum((r.get("bbl_files") or 0 for r in arows)),\n            "total_actual_bytes": sum((r.get("actual_size_bytes") or 0 for r in arows)),\n            "has_empty_no_tex": any(r.get("source_status") in {"pdf_only", "no_latex"} for r in arows),\n            "has_not_found": any(r.get("source_status") == "not_found" for r in arows),\n            "has_unverified": any(not r.get("verified") for r in arows),\n        }\n\n    selected: list[str] = []\n    selected_set: set[str] = set()\n    buckets: dict[str, list[str]] = {}\n\n    def pick(bucket_name: str, candidates: list[str], quota: int) -> None:\n        chosen = []\n        for pid in candidates:\n            if len(selected) >= target_size or len(chosen) >= quota:\n                break\n            if pid not in selected_set:\n                selected_set.add(pid)\n                selected.append(pid)\n                chosen.append(pid)\n        buckets[bucket_name] = chosen\n\n    sorted_ids = sorted(pubs)\n    pick("unverified_or_issue", [p for p in sorted_ids if features[p]["has_unverified"]], 15)\n    pick("not_found_404", [p for p in sorted_ids if features[p]["has_not_found"]], 15)\n    pick("empty_no_tex", [p for p in sorted_ids if features[p]["has_empty_no_tex"]], 25)\n    pick("with_bbl", [p for p in sorted_ids if features[p]["total_bbl_files"] > 0], 35)\n    pick("multi_version_ge3", [p for p in sorted_ids if features[p]["num_versions"] >= 3], 45)\n    pick("multi_version_2", [p for p in sorted_ids if features[p]["num_versions"] == 2], 35)\n    pick("multi_file_tex", [p for p in sorted_ids if features[p]["max_tex_files"] >= 3], 40)\n    top_by_size = sorted(sorted_ids, key=lambda p: (-features[p]["total_actual_bytes"], p))\n    pick("large_tex", top_by_size, 30)\n\n    rng = random.Random(seed)\n    remaining = [p for p in sorted_ids if p not in selected_set]\n    rng.shuffle(remaining)\n    pick("seeded_random_fill", remaining, target_size - len(selected))\n\n    out_dir.mkdir(parents=True, exist_ok=True)\n    pilot_pubs = [pubs[pid] for pid in sorted(selected_set)]\n    pilot_pub_path = out_dir / "publications.parquet"\n    save_publications_parquet(pilot_pubs, str(pilot_pub_path))\n\n    src_ref_path = pub_path.with_name("references.parquet")\n    if src_ref_path.is_file():\n        refs = [r for r in pq.ParquetFile(src_ref_path).read().to_pylist() if r["arxiv_id"] in selected_set]\n    else:\n        refs = []\n    save_references_parquet(refs, str(out_dir / "references.parquet"))\n\n    pilot_course = out_dir / "course"\n    export_to_course_spec(str(out_dir), str(pilot_course))\n\n    selection_info = {\n        "method": "stratified_deterministic_pilot",\n        "seed": seed,\n        "target_size": target_size,\n        "selected_publications": len(selected_set),\n        "bucket_counts": {k: len(v) for k, v in buckets.items()},\n        "buckets": buckets,\n        "selected_ids": sorted(selected_set),\n    }\n    sel_path = out_dir / "pilot_selection.json"\n    atomic_write_json(sel_path, selection_info)\n    return pilot_pub_path, pilot_course, sel_path\n\n\ndef sync_output_batches(parsed_dir: Path, drive_mode_dir: Path, archived_keys: set[str], state_obj: dict) -> int:\n    checkpoint_file = parsed_dir / "checkpoint.json"\n    if not checkpoint_file.is_file():\n        return 0\n    cp = json.loads(checkpoint_file.read_text(encoding="utf-8"))\n    completed = cp.get("completed") or {}\n    new_ids = [pid for pid in sorted(completed) if pid not in archived_keys]\n    batch_dir = drive_mode_dir / "output_batches"\n    batch_dir.mkdir(parents=True, exist_ok=True)\n    if new_ids:\n        state_obj["batch_num"] += 1\n        batch_name = f"batch_{state_obj[\'batch_num\']:04d}.tar.gz"\n        with tempfile.TemporaryDirectory(prefix=".batch-", dir=parsed_dir.parent) as td:\n            local_tar = Path(td) / batch_name\n            added_files = []\n            with tarfile.open(local_tar, "w:gz", compresslevel=1) as tf:\n                for pid in new_ids:\n                    for rel in (completed[pid].get("files") or {}):\n                        src = parsed_dir / rel\n                        if src.is_file():\n                            tf.add(src, arcname=rel, recursive=False)\n                            added_files.append(rel)\n            tar_sha = atomic_copy(local_tar, batch_dir / batch_name)\n        state_obj["batches"].append({\n            "batch_file": batch_name,\n            "sha256": tar_sha,\n            "paper_count": len(new_ids),\n            "file_count": len(added_files),\n        })\n        archived_keys.update(new_ids)\n        atomic_write_json(batch_dir / "batches_manifest.json", state_obj)\n    atomic_copy(checkpoint_file, batch_dir / "checkpoint.json")\n    log_file = parsed_dir / "parser.log"\n    if log_file.is_file():\n        atomic_copy(log_file, drive_mode_dir / "parser.log")\n    return len(new_ids)\n\n\ndef restore_output_batches_if_needed(parsed_dir: Path, drive_mode_dir: Path) -> bool:\n    if (parsed_dir / "checkpoint.json").is_file():\n        return False\n    batch_dir = drive_mode_dir / "output_batches"\n    manifest_file = batch_dir / "batches_manifest.json"\n    cp_file = batch_dir / "checkpoint.json"\n    if not manifest_file.is_file() or not cp_file.is_file():\n        return False\n    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))\n    parsed_dir.mkdir(parents=True, exist_ok=True)\n    for b in manifest.get("batches", []):\n        tar_path = batch_dir / b["batch_file"]\n        if sha256_file(tar_path) != b["sha256"]:\n            raise ValueError(f"Output batch checksum mismatch: {tar_path}")\n        with tarfile.open(tar_path, "r:gz") as tf:\n            for member in tf.getmembers():\n                dest = (parsed_dir / member.name).resolve()\n                if not dest.is_relative_to(parsed_dir.resolve()) or not member.isfile():\n                    raise ValueError(f"Unsafe member in output batch: {member.name}")\n                dest.parent.mkdir(parents=True, exist_ok=True)\n                with tf.extractfile(member) as src_f, dest.open("wb") as out_f:\n                    shutil.copyfileobj(src_f, out_f)\n    atomic_copy(cp_file, parsed_dir / "checkpoint.json")\n    return True\n\n\ndef main() -> None:\n    cfg_path = Path(os.environ.get("M2_RUN_CONFIG", "/content/m2_run_config.json"))\n    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))\n\n    if Path("/dev/nvidia0").exists() or os.environ.get("COLAB_TPU_ADDR"):\n        raise RuntimeError("M2 requires a CPU-only Colab runtime (GPU/TPU detected)")\n\n    mssv = cfg["member_id"]\n    generation = cfg["generation"]\n    mode = cfg["mode"]\n    if mode not in {"pilot", "full", "resume"}:\n        raise ValueError(f"Unsupported mode: {mode}")\n\n    disk_content = shutil.disk_usage("/content")\n    print(\n        f"[runner] Start member={mssv} mode={mode} gen={generation} "\n        f"python={sys.version.split()[0]} cpu_count={os.cpu_count()} "\n        f"free_disk_gb={disk_content.free / 1024**3:.2f} COLAB_RELEASE_TAG={os.environ.get(\'COLAB_RELEASE_TAG\')}",\n        flush=True,\n    )\n\n    source_root, source_manifest = install_bundle(cfg["bundle_path"], "/content/m2_source", cfg["bundle_sha256"])\n    marker = source_root / ".editable_installed"\n    if not marker.is_file():\n        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(source_root)], check=True)\n        marker.write_text("ok\\n", encoding="utf-8")\n    os.chdir(source_root)\n    if str(source_root) not in sys.path:\n        sys.path.insert(0, str(source_root))\n\n    from src.crawler.drive_backup import CheckpointBackup\n    from src.crawler.main import Checkpoint, _materialize\n    from src.crawler.observations import save_observations\n    from src.crawler.storage import export_to_course_spec, save_publications_parquet, save_references_parquet\n    from src.crawler.validate_sources import audit_sources, validate_course_tables\n    import pyarrow.parquet as pq\n\n    work_dir = Path(f"/content/m2_work/{mssv}/{generation}")\n    verify_path = work_dir / ".restore_verified.json"\n    restore_sec = 0.0\n    restore_reused = False\n    restore_source = "existing_local"\n\n    if (\n        verify_path.is_file()\n        and (work_dir / "shard/_checkpoint/state.json").is_file()\n        and json.loads(verify_path.read_text(encoding="utf-8")).get("generation") == generation\n        and json.loads(verify_path.read_text(encoding="utf-8")).get("head_sha256") == cfg.get("head_sha256")\n    ):\n        restore_reused = True\n        print(f"[runner] Reusing already-verified restored corpus at {work_dir}", flush=True)\n    else:\n        if disk_content.free < 15 * 1024**3:\n            raise OSError(f"Insufficient free disk on /content before restore: {disk_content.free / 1024**3:.2f} GiB")\n        backup_dir, restore_source = ensure_backup_dir(cfg)\n        head = json.loads((backup_dir / "HEAD.json").read_text(encoding="utf-8"))\n        if head.get("generation") != generation:\n            raise ValueError(f"HEAD mismatch: expected {generation}, got {head.get(\'generation\')}")\n        if cfg.get("head_sha256") and head.get("sha256") != cfg["head_sha256"]:\n            raise ValueError(f"HEAD receipt sha256 mismatch: expected {cfg[\'head_sha256\']}, got {head.get(\'sha256\')}")\n        if work_dir.exists():\n            shutil.rmtree(work_dir)\n        t_res = time.perf_counter()\n        cb = CheckpointBackup(work_dir / "raw", work_dir / "shard", backup_dir)\n        ok = cb.restore(work_dir)\n        restore_sec = time.perf_counter() - t_res\n        if not ok:\n            raise RuntimeError(f"CheckpointBackup.restore() returned False for {mssv}")\n        atomic_write_json(verify_path, {\n            "member_id": mssv,\n            "generation": generation,\n            "head_sha256": head.get("sha256"),\n            "restore_source": restore_source,\n            "restore_wall_sec": restore_sec,\n            "restored_at_utc": datetime.now(timezone.utc).isoformat(),\n        })\n        print(f"[runner] Restore completed in {restore_sec:.2f}s via {restore_source}", flush=True)\n\n    shard_dir = work_dir / "shard"\n    state_json = json.loads((shard_dir / "_checkpoint/state.json").read_text(encoding="utf-8"))\n    done_ids_count = len(state_json.get("done_ids") or [])\n    pub_parquet = shard_dir / "publications.parquet"\n    ref_parquet = shard_dir / "references.parquet"\n    materialized_flag = False\n    need_materialize = not pub_parquet.is_file() or not ref_parquet.is_file()\n    if not need_materialize:\n        existing_rows = pq.ParquetFile(pub_parquet).metadata.num_rows\n        if existing_rows < done_ids_count:\n            need_materialize = True\n    if need_materialize:\n        cp = Checkpoint(str(shard_dir))\n        cp.load(state_json["member_id"], state_json["id_range"], backup_legacy=False)\n        pubs_map, refs_map, obs_map = _materialize(cp)\n        save_publications_parquet([pubs_map[k] for k in sorted(pubs_map)], str(pub_parquet))\n        all_refs = [r for k in sorted(refs_map) for r in refs_map[k]]\n        save_references_parquet(all_refs, str(ref_parquet))\n        save_observations([obs_map[k] for k in sorted(obs_map)], str(shard_dir / "crawl_status.parquet"))\n        materialized_flag = True\n        print(f"[runner] Materialized working tables from checkpoint: {len(pubs_map)} pubs, {len(all_refs)} refs", flush=True)\n\n    course_dir = work_dir / "course"\n    need_course = not (course_dir / "papers.parquet").is_file() or not (course_dir / "references.parquet").is_file()\n    if not need_course:\n        c_rows = pq.ParquetFile(course_dir / "papers.parquet").metadata.num_rows\n        w_rows = pq.ParquetFile(pub_parquet).metadata.num_rows\n        if c_rows != w_rows:\n            need_course = True\n    if need_course:\n        export_to_course_spec(str(shard_dir), str(course_dir))\n    course_check = validate_course_tables(str(course_dir))\n    if not course_check["valid"]:\n        raise RuntimeError(f"Course export invalid for {mssv}: {course_check}")\n\n    audit_dir = work_dir / "audit"\n    audit_parquet = audit_dir / "source_audit.parquet"\n    audit_json = audit_dir / "source_audit.json"\n    audit_sec = 0.0\n    if not audit_json.is_file() or not audit_parquet.is_file():\n        t_aud = time.perf_counter()\n        audit_report = audit_sources(str(pub_parquet), str(work_dir / "raw"), str(audit_parquet))\n        audit_sec = time.perf_counter() - t_aud\n        print(f"[runner] Source audit finished in {audit_sec:.2f}s: {audit_report[\'reasons\']}", flush=True)\n    else:\n        audit_report = json.loads(audit_json.read_text(encoding="utf-8"))\n\n    drive_root = Path(cfg.get("drive_output_root", "/content/drive/MyDrive/arxiv-harvester-m2"))\n    drive_mode_dir = drive_root / "members" / mssv / generation / ("full" if mode == "resume" else mode)\n    drive_mode_dir.mkdir(parents=True, exist_ok=True)\n\n    if mode == "pilot":\n        pilot_dir = work_dir / "pilot_input"\n        pub_input, course_input, sel_path = select_stratified_pilot(\n            pub_parquet,\n            audit_parquet,\n            int(cfg.get("pilot_size", 250)),\n            int(cfg.get("pilot_seed", 20261121)),\n            pilot_dir,\n        )\n        parsed_dir = work_dir / "parsed_pilot"\n        if parsed_dir.exists():\n            shutil.rmtree(parsed_dir)\n    else:\n        pub_input, course_input, sel_path = pub_parquet, course_dir, None\n        parsed_dir = work_dir / "parsed_full"\n        if mode == "full" and cfg.get("fresh_output", True) and parsed_dir.exists():\n            shutil.rmtree(parsed_dir)\n        elif mode == "resume":\n            restored_batches = restore_output_batches_if_needed(parsed_dir, drive_root / "members" / mssv / generation / "full")\n            if restored_batches:\n                print("[runner] Restored prior output batches from Drive for --resume", flush=True)\n\n    parsed_dir.mkdir(parents=True, exist_ok=True)\n    time_bin = "/usr/bin/time" if Path("/usr/bin/time").is_file() else shutil.which("time")\n    parser_cmd = [\n        sys.executable,\n        "-u",\n        "-m",\n        "src.parser.main",\n        "--publications",\n        str(pub_input),\n        "--raw-dir",\n        str(work_dir / "raw"),\n        "--output-dir",\n        str(parsed_dir),\n        "--sample-interval-sec",\n        str(cfg.get("sample_interval_sec", 0.5)),\n        "--min-free-gb",\n        str(cfg.get("min_free_gb", 2.0)),\n    ]\n    if mode == "resume" and (parsed_dir / "checkpoint.json").is_file():\n        parser_cmd.append("--resume")\n    if cfg.get("root_overrides_rel"):\n        ov_path = source_root / cfg["root_overrides_rel"]\n        if ov_path.is_file():\n            parser_cmd.extend(["--root-overrides", str(ov_path)])\n    command = ([time_bin, "-v", "-o", str(parsed_dir / "time_v.txt"), *parser_cmd]\n               if time_bin else parser_cmd)\n\n    run_config_doc = {\n        "member_id": mssv,\n        "short_name": cfg.get("short_name"),\n        "generation": generation,\n        "head_sha256": cfg.get("head_sha256"),\n        "mode": mode,\n        "bundle_sha256": cfg["bundle_sha256"],\n        "source_snapshot_sha256": source_manifest.get("source_snapshot_sha256"),\n        "source_base_commit": source_manifest.get("source_base_commit"),\n        "command": command,\n        "started_at_utc": datetime.now(timezone.utc).isoformat(),\n    }\n    atomic_write_json(parsed_dir / "run_configuration.json", run_config_doc)\n\n    archived_keys: set[str] = set()\n    batch_state = {"batch_num": 0, "batches": []}\n    if mode == "resume":\n        old_bm = drive_root / "members" / mssv / generation / "full" / "output_batches" / "batches_manifest.json"\n        if old_bm.is_file():\n            batch_state = json.loads(old_bm.read_text(encoding="utf-8"))\n        cp_existing = parsed_dir / "checkpoint.json"\n        if cp_existing.is_file():\n            archived_keys.update((json.loads(cp_existing.read_text(encoding="utf-8")).get("completed") or {}).keys())\n\n    stop_sync = threading.Event()\n    sync_interval = float(cfg.get("sync_batch_interval_sec", 120.0))\n    sync_target = drive_root / "members" / mssv / generation / ("full" if mode == "resume" else mode)\n\n    def _sync_worker() -> None:\n        while not stop_sync.wait(sync_interval):\n            try:\n                sync_output_batches(parsed_dir, sync_target, archived_keys, batch_state)\n            except Exception as exc:  # noqa: BLE001\n                print(f"[runner] Periodic Drive sync warning: {exc}", flush=True)\n\n    sync_thread = threading.Thread(target=_sync_worker, daemon=True)\n    sync_thread.start()\n\n    t_parse = time.perf_counter()\n    with (parsed_dir / "parser.log").open("w", encoding="utf-8") as log:\n        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n        assert process.stdout is not None\n        for line in process.stdout:\n            print(line, end="", flush=True)\n            log.write(line)\n            log.flush()\n        exit_code = process.wait()\n    parser_wall_sec = time.perf_counter() - t_parse\n    child_rusage = resource.getrusage(resource.RUSAGE_CHILDREN)\n    os_children_peak_rss_kb = int(child_rusage.ru_maxrss)\n\n    stop_sync.set()\n    sync_thread.join(timeout=30)\n    if exit_code != 0:\n        sync_output_batches(parsed_dir, sync_target, archived_keys, batch_state)\n        raise RuntimeError(f"Parser failed with exit code {exit_code}")\n\n    t_val = time.perf_counter()\n    val_proc = subprocess.run(\n        [\n            sys.executable,\n            "-m",\n            "src.parser.validation",\n            "--course-dir",\n            str(course_input),\n            "--parsed-dir",\n            str(parsed_dir),\n            "--publications",\n            str(pub_input),\n            "--output",\n            str(parsed_dir / "project_validation.json"),\n        ],\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    validate_wall_sec = time.perf_counter() - t_val\n    if val_proc.returncode not in (0, 2):\n        raise RuntimeError(f"Validation failed ({val_proc.returncode}): {val_proc.stdout}\\n{val_proc.stderr}")\n\n    t_sync = time.perf_counter()\n    sync_output_batches(parsed_dir, sync_target, archived_keys, batch_state)\n    stats = json.loads((parsed_dir / "stats.json").read_text(encoding="utf-8"))\n    validation = json.loads((parsed_dir / "project_validation.json").read_text(encoding="utf-8"))\n\n    runner_summary = {\n        "member_id": mssv,\n        "short_name": cfg.get("short_name"),\n        "generation": generation,\n        "mode": mode,\n        "restore_reused": restore_reused,\n        "restore_source": restore_source,\n        "restore_wall_sec": restore_sec,\n        "materialized_from_checkpoint": materialized_flag,\n        "checkpoint_done_ids": done_ids_count,\n        "working_publications": pq.ParquetFile(pub_parquet).metadata.num_rows,\n        "source_audit_wall_sec": audit_sec,\n        "source_audit_summary": audit_report,\n        "parser_wall_sec": parser_wall_sec,\n        "os_children_peak_rss_kb": os_children_peak_rss_kb,\n        "time_binary_used": time_bin,\n        "validate_wall_sec": validate_wall_sec,\n        "parser_status": stats.get("status"),\n        "parser_runtime_sec": stats.get("runtime_sec"),\n        "status_counts": stats.get("status_counts"),\n        "structural_valid": validation.get("structural_valid"),\n        "processing_complete": validation.get("processing_complete"),\n        "environment": stats.get("environment"),\n        "finished_at_utc": datetime.now(timezone.utc).isoformat(),\n    }\n    atomic_write_json(parsed_dir / "runner_summary.json", runner_summary)\n\n    # Sync summary & log artifacts to Drive\n    sync_files: dict[str, str] = {}\n    for fname in ("stats.json", "manifest.json", "project_validation.json", "time_v.txt", "parser.log", "run_configuration.json", "runner_summary.json"):\n        src = parsed_dir / fname\n        if src.is_file():\n            sync_files[fname] = atomic_copy(src, drive_mode_dir / fname)\n    for run_file in sorted((parsed_dir / "runs").glob("*/*")):\n        if run_file.name in {"stats.json", "resource_samples.jsonl"} and run_file.is_file():\n            rel = run_file.relative_to(parsed_dir).as_posix()\n            sync_files[rel] = atomic_copy(run_file, drive_mode_dir / rel)\n    if sel_path and sel_path.is_file():\n        sync_files["pilot_selection.json"] = atomic_copy(sel_path, drive_mode_dir / "pilot_selection.json")\n    member_drive_dir = drive_root / "members" / mssv / generation\n    for afname in ("source_audit.json", "source_audit.retry.jsonl", "source_audit.parquet"):\n        src = audit_dir / afname\n        if src.is_file():\n            sync_files[f"audit/{afname}"] = atomic_copy(src, member_drive_dir / "audit" / afname)\n    if mode != "pilot":\n        for src, rel in (\n            (shard_dir / "publications.parquet", "m1_tables/publications.parquet"),\n            (shard_dir / "references.parquet", "m1_tables/references.parquet"),\n            (shard_dir / "crawl_status.parquet", "m1_tables/crawl_status.parquet"),\n            (shard_dir / "stats.json", "m1_tables/stats.json"),\n            (course_dir / "papers.parquet", "m1_tables/course/papers.parquet"),\n            (course_dir / "references.parquet", "m1_tables/course/references.parquet"),\n        ):\n            if src.is_file():\n                sync_files[rel] = atomic_copy(src, member_drive_dir / rel)\n    runner_summary["drive_sync_wall_sec"] = time.perf_counter() - t_sync\n    atomic_write_json(parsed_dir / "runner_summary.json", runner_summary)\n    sync_files["runner_summary.json"] = atomic_copy(parsed_dir / "runner_summary.json", drive_mode_dir / "runner_summary.json")\n    atomic_write_json(drive_mode_dir / "sync_manifest.json", {"files": sync_files, "synced_at_utc": datetime.now(timezone.utc).isoformat()})\n\n    print(\n        f"[runner] DONE member={mssv} mode={mode} status={stats.get(\'status\')} "\n        f"structural_valid={validation.get(\'structural_valid\')} "\n        f"runtime_sec={stats.get(\'runtime_sec\'):.2f} statuses={stats.get(\'status_counts\')}",\n        flush=True,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
subprocess.run([sys.executable, '-u', str(runner_path)], check=True)


In [ ]:
import json, os
from pathlib import Path

cfg = json.loads(Path(os.environ.get('M2_RUN_CONFIG', '/content/m2_run_config.json')).read_text(encoding='utf-8'))
work = Path(f"/content/m2_work/{cfg['member_id']}/{cfg['generation']}")
out = work / ('parsed_pilot' if cfg['mode'] == 'pilot' else 'parsed_full')
for name in ('runner_summary.json', 'project_validation.json'):
    p = out / name
    if p.is_file():
        print('===', name, '===')
        print(p.read_text(encoding='utf-8')[:4000])
